# ~1B GGUF baseline on a T4

Runs **Qwen3.5-0.8B quantized to a GGUF** (`Q4_K_M` or `Q8_0`, set in *Settings* below), untuned, through `llama-server`, on the `dev` suite, with the same harness and settings as `qwen3.5-0.8b-fp16` and the teacher `qwen3.8-27b-fp8-v5`.

These are the **shippable-size** scores for the 0.8B, served by llama.cpp, the runtime family an iPhone build would use: `Q4_K_M` is about 0.50 GiB of weights and `Q8_0` about 0.8 GB, against 1.40 GiB at fp16. `Q8_0` is close to lossless, so it runs on the same server as `Q4_K_M` to separate what the 4-bit weights cost from what the change of server costs. The tuned model's export will be judged against these rows.

| | fp16 baseline | this run |
|---|---|---|
| model | Qwen3.5-0.8B | Qwen3.5-0.8B |
| weights | fp16 | the `QUANT` GGUF, converted here from the Hugging Face checkpoint (no MTP head, no vision tower) |
| server | vLLM | `llama-server`, llama.cpp release `b12400`, CUDA |
| thinking | none | none |
| temperature | 0 | 0 |
| tool rounds / tool tokens / answer tokens | 10 / 800 / 1500 | 10 / 800 / 1500 (the defaults) |
| cases in flight | 16 | 16 (`-np 16`, 16,384 tokens per slot like vLLM's `--max-model-len`) |
| suite | `dev`, 280 cases | `dev`, 280 cases |

**A tuned model goes through the same cells.** Point `SOURCE` at its merged Hugging Face folder (the one `sft.ipynb` leaves on Drive, or a `.zip` of it uploaded to `/content`) and name it with `NAME`; the run is then `<NAME>-<quant>`, e.g. `qwen3.5-0.8b-sft-r1-q8_0`, and its score sits beside the untuned rows above at the same quantization, through the same server.

Use **Runtime → Change runtime type → T4 GPU**. The notebook clones `REPO_URL` at `BRANCH`, [kurexi/cost](https://github.com/kurexi/cost) by default; point them at a fork to run your own changes (for a private one, add a Colab secret named `GITHUB_TOKEN` with read access to it). `AGENT` is the agent's folder inside the clone: empty for kurexi/cost, where the agent is the whole repository. `RECORDS` is the folder, inside the clone, that holds `runs/` and `RESULTS.md`; `eval-records` by default, the repository's own runs; leave it empty to keep them in the agent's `eval/` folder. The eval scripts find it through `SPENDING_EVAL_RECORDS`.

Time: llama.cpp comes as prebuilt binaries when the runtime's GPU driver can run them, a few minutes; otherwise the notebook builds it, 15–30 minutes. The run itself should take well under half an hour.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None):
    """Run a command with its output in the cell: a process started by
    subprocess.run prints to the runtime's log, which Colab doesn't show."""
    process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "main"
RECORDS = "eval-records"
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

- `SOURCE`: the model to convert. A Hugging Face id (`"Qwen/Qwen3.5-0.8B"`, the untuned baseline), a folder (a merged model on Drive, e.g. `"/content/drive/MyDrive/spending-agent/models/qwen3.5-0.8b-sft-r1"`; Drive is mounted when the path is on it), or a `.zip` of such a folder uploaded to `/content`.
- `NAME`: what the model is called in run names: `"qwen3.5-0.8b"` for the base, `"qwen3.5-0.8b-sft-r1"` for the first SFT round.
- `QUANT`: `"Q4_K_M"`, `"Q5_K_M"`, `"Q6_K"` or `"Q8_0"`.
- `SUITES`: `dev`, and `dev-novel`, where every merchant and item name is invented and none can have been trained on. A model that reads receipts scores about the same on both; one that remembers shops drops on `dev-novel`. For a base row that already has its `dev` run, keep only `"dev-novel"`.

The `dev` run is named `<NAME>-<quant>` (`qwen3.5-0.8b-q4_k_m`, `qwen3.5-0.8b-sft-r1-q8_0`) and the `dev-novel` one `<NAME>-<quant>-dev-novel`, so two models or two quantizations can never land in one run folder. The convert cell stops before the run if the file is over 1 GB, the size past which a quantization isn't worth shipping.

In [ ]:
SOURCE = "Qwen/Qwen3.5-0.8B"
NAME = "qwen3.5-0.8b"
QUANT = "Q8_0"
SUITES = ["dev", "dev-novel"]
RUN = f"{NAME}-{QUANT.lower()}"
RUNS = {suite: RUN if suite == "dev" else f"{RUN}-{suite}" for suite in SUITES}
SIZE_LIMIT_BYTES = 10**9
if SOURCE.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
print(RUNS)

## Get llama.cpp

Release `b12400`, a build whose converter knows Qwen3.5 (`Qwen3_5ForConditionalGeneration`). The source is fetched for the converter (seconds; nothing is compiled), and the server comes from the release's **prebuilt** Linux CUDA 12.8 binaries plus the CUDA runtime libraries the release ships beside them.

A T4 is Turing (`sm_75`), and the release carries only PTX for it, not compiled kernels: the driver compiles them on first use. That needs a driver from the 570 series or later, so the cell checks the driver, then asks the server which devices it sees. If either check fails, or the download does, it builds from source instead (15–30 minutes, only `llama-server` and `llama-quantize`, for `sm_75` only). It prints which path it took.

The first server start and the first requests are slower while the driver compiles kernels; `CUDA_CACHE_MAXSIZE` is raised so they're compiled once per runtime, not on every restart.

In [ ]:
LLAMA_TAG = "b12400"
LLAMA = "/content/llama.cpp"
ASSET = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64"
PREBUILT = f"/content/llama-{LLAMA_TAG}"
RELEASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
os.environ["CUDA_CACHE_MAXSIZE"] = str(4 << 30)

if not os.path.exists(f"{LLAMA}/convert_hf_to_gguf.py"):
    sh(["rm", "-rf", LLAMA])
    os.makedirs(LLAMA)
    for step in (["git", "init", "-q"],
                 ["git", "remote", "add", "origin", "https://github.com/ggml-org/llama.cpp"],
                 ["git", "fetch", "-q", "--depth", "1", "origin", "tag", LLAMA_TAG],
                 ["git", "checkout", "-q", LLAMA_TAG]):
        sh(step, cwd=LLAMA)


def driver_major():
    out = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout
    return int(out.strip().split(".")[0])


def sees_cuda(bin_dir):
    out = subprocess.run([f"{bin_dir}/llama-server", "--list-devices"], capture_output=True, text=True)
    return "CUDA0" in out.stdout + out.stderr


def prebuilt():
    if driver_major() < 570:
        print(f"driver {driver_major()} can't compile CUDA 12.8 PTX for the T4")
        return None
    try:
        if not os.path.exists(f"{PREBUILT}/llama-server"):
            for name in (ASSET, f"cudart-{ASSET}"):
                sh(["curl", "-fsSL", "-o", f"/content/{name}.tar.gz", f"{RELEASE}/{name}.tar.gz"])
                sh(["tar", "-xzf", f"/content/{name}.tar.gz", "-C", "/content"])
            subprocess.run(f"cp -a /content/cudart-{ASSET}/. {PREBUILT}/", shell=True, check=True)
    except subprocess.CalledProcessError as error:
        print(f"the release download failed: {error}")
        return None
    if not sees_cuda(PREBUILT):
        print("the prebuilt server doesn't see the GPU")
        return None
    return PREBUILT


def built():
    if not os.path.exists(f"{LLAMA}/build/bin/llama-server"):
        sh(["cmake", "-B", "build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
                        "-DLLAMA_OPENSSL=OFF", "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"],
                       cwd=LLAMA)
        sh(["cmake", "--build", "build", "--config", "Release", "-j", str(os.cpu_count()),
                        "--target", "llama-server", "llama-quantize"], cwd=LLAMA)
    if not sees_cuda(f"{LLAMA}/build/bin"):
        raise SystemExit("the built server doesn't see the GPU either: check Runtime → Change runtime type")
    return f"{LLAMA}/build/bin"


BIN = prebuilt()
BINARIES = "release" if BIN else "built from source"
BIN = BIN or built()
LLAMA_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=LLAMA, capture_output=True, text=True).stdout.strip()
print(f"llama.cpp {LLAMA_TAG} ({LLAMA_COMMIT[:7]}), {BINARIES}: {BIN}")

## Convert and quantize

The converter runs in its own virtual environment with llama.cpp's pinned requirements (CPU PyTorch, `transformers` 4.57.6), so it can't clash with Colab's packages. A folder or zip is used as it is; a Hugging Face id is downloaded. The checkpoint is bf16, so the intermediate GGUF is bf16 too (an f16 intermediate can overflow); it's kept, so switching `QUANT` only reruns the quantize step. `--no-mtp` leaves out the multi-token-prediction head, which the fp16 baseline didn't use either; the vision tower is never part of the text GGUF.

In [ ]:
import glob

BF16 = f"/content/models/{NAME}-bf16.gguf"
GGUF = f"/content/models/{RUN}.gguf"
VENV = "/content/convert-venv"
os.makedirs("/content/models", exist_ok=True)
MODEL_DIR = SOURCE if os.path.isdir(SOURCE) else f"/content/models/{NAME}-hf"
if SOURCE.endswith(".zip"):
    if not os.path.exists(MODEL_DIR):
        sh(["unzip", "-q", SOURCE, "-d", MODEL_DIR])
    MODEL_DIR = os.path.dirname(sorted(glob.glob(f"{MODEL_DIR}/**/config.json", recursive=True), key=len)[0])

if not os.path.exists(BF16):
    if not os.path.exists(f"{VENV}/bin/python"):
        sh(["pip", "install", "-q", "uv"])
        sh(["uv", "venv", "-q", "--python", "3.12", VENV])
    sh(["uv", "pip", "install", "-q", "--python", f"{VENV}/bin/python", "huggingface_hub",
                    "-r", f"{LLAMA}/requirements/requirements-convert_hf_to_gguf.txt"])
    if not os.path.isdir(MODEL_DIR):
        sh([f"{VENV}/bin/python", "-c",
                        f"from huggingface_hub import snapshot_download; snapshot_download('{SOURCE}', local_dir='{MODEL_DIR}')"])
    sh([f"{VENV}/bin/python", f"{LLAMA}/convert_hf_to_gguf.py", MODEL_DIR,
                    "--outtype", "bf16", "--no-mtp", "--outfile", BF16])
if not os.path.exists(GGUF):
    sh([f"{BIN}/llama-quantize", BF16, GGUF, QUANT])
for path in (BF16, GGUF):
    print(f"{os.path.getsize(path) / 1e9:.3f} GB  {os.path.getsize(path) / 2**30:.3f} GiB  {path}")
if os.path.getsize(GGUF) > SIZE_LIMIT_BYTES:
    raise SystemExit(f"{QUANT} is {os.path.getsize(GGUF) / 1e9:.2f} GB, over the 1 GB limit: not worth running")

## Start the server

Served as `qwen`, like the other baselines, with every layer on the GPU and 16 parallel slots of 16,384 tokens each. Tool calling and `chat_template_kwargs` (how the agent turns thinking off) need the Jinja chat template, which is the default in this build. Each session resends its whole conversation every round (about 16K prompt tokens per case over six rounds in the fp16 run), so the server reusing a slot's earlier prefix is most of the speed. Qwen3.5 is a hybrid whose DeltaNet layers can't be rewound, so that reuse runs on context checkpoints: copies of the recurrent state, about 19 MiB each, kept in host RAM. The default of 32 per slot would grow to about 10 GiB across 16 slots, near all of a T4 runtime's RAM; `-ctxcp 4` keeps the latest four per slot (about 1.2 GiB), enough for a session's last turns. Logs go to `/content/llama-server.log`. The cell first stops any server left from an earlier attempt, and refuses to start without the GGUF. The next cell waits for the server and prints its latest log line every 30 seconds; if the server exits, it stops at once and shows the end of the log.

In [ ]:
import time

subprocess.run('pkill -f "[l]lama-server"', shell=True)
time.sleep(3)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
if not os.path.exists(GGUF):
    raise SystemExit(f"{GGUF} doesn't exist: run the convert cell first")
SERVER_LOG = "/content/llama-server.log"
server = subprocess.Popen([f"{BIN}/llama-server", "-m", GGUF, "--alias", "qwen", "-ngl", "all", "-np", "16",
                           "-c", "262144", "--jinja", "-cram", "2048", "-ctxcp", "4", "--host", "127.0.0.1",
                           "--port", "8090"],
                          stdout=open(SERVER_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

In [ ]:
import urllib.request


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def wait_for_server(process, log_path, limit_seconds=1200):
    started = time.time()
    reported = started
    while True:
        if process.poll() is not None:
            raise SystemExit(f"the server exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"server is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > limit_seconds:
            raise SystemExit(f"the server isn't up after {limit_seconds} s:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)


wait_for_server(server, SERVER_LOG)

## Check the harness and the answer format

`oracle` must grade 1.000 and `silent` 0. The probe must accept the **tool round** and the **full bounded schema**: a run that falls back to a looser format isn't comparable with the fp16 and teacher rows, which answered every case on `json_schema`. The cell stops if either is refused; the probe's output and `/content/llama-server.log` say why.

Then a smoke run of 8 cases (under `smoke-…`, which git ignores) shows that whole sessions go through before the full run.

In [ ]:
%cd {EVAL}
!python3 run.py --suite dev --name check-oracle --backend oracle --restart | grep "^record_core"
!python3 run.py --suite dev --name check-silent --backend silent --restart | grep "^record_core"
probe = subprocess.run(["python3", "probe_formats.py", "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen"],
                       capture_output=True, text=True)
print(probe.stdout, probe.stderr)
verdicts = dict((line[:32].strip(), line[32:].strip()) for line in probe.stdout.splitlines() if line.strip())
for required in ("tool round (tool_choice auto)", "full bounded schema"):
    if not verdicts.get(required, "").startswith("accepted"):
        raise SystemExit(f"the server refused the {required}: not a fair baseline, stop here")

In [ ]:
%cd {EVAL}
!python3 run.py --suite dev --name smoke-{RUN} --base-url http://127.0.0.1:8090/v1 --model qwen --reasoning-effort none --workers 8 --limit 8 --restart
!python3 grade.py smoke-{RUN} | head -20

## The run

If Colab disconnects, rerun the llama.cpp, convert and server cells (they skip what's already there if the runtime survived) and then this cell: each run resumes from its last finished case. It runs the suites in `SUITES` one after the other, about 20 minutes each.

In [ ]:
%cd {EVAL}
for suite, run in RUNS.items():
    sh(["python3", "run.py", "--suite", suite, "--name", run, "--base-url", "http://127.0.0.1:8090/v1",
                    "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"])

## What was served

`meta.json` records the harness settings but not the file behind `qwen`, and a quantized model's score is the score of that exact file. So `server.json` records it: the llama.cpp release and commit, whether its binaries were the release's or built here, the quantization, the GGUF's size and SHA-256, and the server's flags. For a tuned model it also carries the merge report (base, adapter, rank) from the merged folder.

In [ ]:
import hashlib
import json

digest = hashlib.sha256()
with open(GGUF, "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        digest.update(block)
served = {
    "source": SOURCE,
    "llama_cpp_tag": LLAMA_TAG,
    "llama_cpp_commit": LLAMA_COMMIT,
    "llama_cpp_binaries": BINARIES,
    "convert": "convert_hf_to_gguf.py --outtype bf16 --no-mtp",
    "quantization": QUANT,
    "gguf_bytes": os.path.getsize(GGUF),
    "gguf_gib": round(os.path.getsize(GGUF) / 2**30, 3),
    "gguf_sha256": digest.hexdigest(),
    "server": "llama-server -ngl all -np 16 -c 262144 --jinja -cram 2048 -ctxcp 4",
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
}
if os.path.exists(f"{MODEL_DIR}/merge.json"):
    with open(f"{MODEL_DIR}/merge.json") as f:
        served["merge"] = json.load(f)
for run in RUNS.values():
    with open(f"{RECORDS_DIR}/runs/{run}/server.json", "w") as f:
        json.dump(served, f, indent=1)
print(json.dumps(served, indent=1))

## Against the fp16 baseline

fp16 is the first run, so the deltas read as the cost of quantizing (and, for a tuned model, what tuning changed on top). For the base model the other committed GGUF runs sit between; for a tuned model, the untuned run at the same quantization and the tuned model's bf16 run (from `sft.ipynb`) do, when they are committed. The cases fixed and broken are counted between fp16 and this run. On `dev-novel` the teacher's run comes first, then the untuned model at the same quantization and the tuned model's bf16 run when they are committed. `board.py --write` adds the new rows to `RESULTS.md`, one table per suite.

In [ ]:
%cd {EVAL}
if NAME == "qwen3.5-0.8b":
    between = ["qwen3.5-0.8b-q4_k_m", "qwen3.5-0.8b-q8_0"]
else:
    between = [f"qwen3.5-0.8b-{QUANT.lower()}", f"{NAME}-bf16"]
lineups = {"dev": ["qwen3.5-0.8b-fp16", *between],
           "dev-novel": ["qwen3.8-27b-fp8-v5-dev-novel", *(f"{r}-dev-novel" for r in between)]}
for suite, run in RUNS.items():
    others = [r for r in lineups[suite] if r != run and os.path.exists(f"{RECORDS_DIR}/runs/{r}/results.jsonl")]
    if others:
        sh(["python3", "compare.py", *others, run])
!python3 board.py --write

## Take the results home

Download the zip, then unpack it into your records folder and commit it: `runs/<run>/` (with `server.json`) and the updated `RESULTS.md`. Add a line for the run under *Notes per run*, with the GGUF size from `server.json` (and, for a tuned model, the round and its training settings).

In [ ]:
%cd {RECORDS_DIR}
sh(["zip", "-qr", f"/content/{RUN}.zip", *(f"runs/{run}" for run in RUNS.values()), "RESULTS.md"])
from google.colab import files
files.download(f"/content/{RUN}.zip")